In [1]:
# %pip install pandas numpy matplotlib scikit-learn StandardScaler

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [3]:
# /content/drive/MyDrive/cricket_project/IPL_B2B_Dataset.csv
# read the following file from drive in pd

df = pd.read_csv('IPL_B2B_Dataset.csv')
len(df)
df


,Index,batting_team,bowling_team,city,runs_left,balls_left,wickets,target_runs,cur_run_rate,req_run_rate,result
0,0,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,207,119,10,208,6.000000,10.43697479,0
1,1,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,207,118,10,208,3.000000,10.52542373,0
2,2,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,207,117,10,208,2.000000,10.61538462,0
3,3,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,205,116,10,208,4.500000,10.60344828,0
4,4,Royal Challengers Bangalore,Sunrisers Hyderabad,Hyderabad,201,115,10,208,8.400000,10.48695652,0
...,...,...,...,...,...,...,...,...,...,...,...
72408,72408,Chennai Super Kings,Mumbai Indians,Hyderabad,1,4,5,153,7.862069,1.5,0
72409,72409,Chennai Super Kings,Mumbai Indians,Hyderabad,-1,3,5,153,7.897436,-2,0
72410,72410,Chennai Super Kings,Mumbai Indians,Hyderabad,-2,2,4,153,7.881356,-6,0
72411,72411,Chennai Super Kings,Mumbai Indians,Hyderabad,-4,1,4,153,7.915966,-24,0


In [4]:
import numpy as np
import pandas as pd

# --- columns we expect ---
# runs_left, balls_left, wickets, target_runs, cur_run_rate, req_run_rate, match_id, inning (optional)

# make numeric (invalid -> NaN)
cols_num = ['runs_left','balls_left','wickets','target_runs','cur_run_rate','req_run_rate']
df[cols_num] = df[cols_num].apply(pd.to_numeric, errors='coerce')

# 1) remove rows with impossible negative balls (these are likely bad rows)
df = df[df['balls_left'].notna()]            # remove NaN balls
df = df[df['balls_left'] >= 0]               # drop negative balls

# 2) runs_left: if negative -> set to 0 (chase finished). Do not take abs().
df['runs_left'] = df['runs_left'].fillna(0)
df.loc[df['runs_left'] < 0, 'runs_left'] = 0

# 3) wickets: keep only 0..10
df = df[df['wickets'].notna()]
df = df[(df['wickets'] >= 0) & (df['wickets'] <= 10)]

# 4) Limit balls_left to a reasonable cap per match type.
#    Heuristic: if target_runs suggests T20 (<= 250) use 120, else ODI use 300, else 600 cap.
def infer_cap(tr):
    if pd.isna(tr):
        return 600
    if tr <= 250:
        return 120
    if tr <= 450:
        return 300
    return 600

# apply per-row cap
caps = df['target_runs'].apply(infer_cap)
df = df[df['balls_left'] <= caps]

# 5) Recompute required run rate safely (runs per over equivalent)
#    avoid divide-by-zero: use a tiny epsilon, or set req_rr = large number if no balls left but runs remain
eps = 1e-9
df['req_run_rate'] = df['runs_left'] / ((df['balls_left'] / 6.0) + eps)

# 6) Clean remaining NaN / inf: replace inf with NaN then fill with median (not a magic constant)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

# 7) Optional: remove rows that still look impossible after cleaning
#    e.g., extremely large req_run_rate ( > 100 ) or negative target_runs
df = df[df['req_run_rate'] <= 100]            # tune threshold if you want
df = df[df['target_runs'].notna() & (df['target_runs'] > 0)]

# show counts / quick sanity
print("Rows after cleaning:", len(df))
print("runs_left min/max:", df['runs_left'].min(), df['runs_left'].max())
print("balls_left min/max:", df['balls_left'].min(), df['balls_left'].max())
print("req_run_rate min/max:", df['req_run_rate'].min(), df['req_run_rate'].max())


Rows after cleaning: 71888
runs_left min/max: 0 250
balls_left min/max: 0 119
req_run_rate min/max: 0.0 99.99999980000001


In [5]:
# Detect new match
new_match = (
    (df['runs_left'].diff() > 0) |
    (df['batting_team'] != df['batting_team'].shift()) |
    (df['bowling_team'] != df['bowling_team'].shift())
)

# Assign match_id
df['match_id'] = new_match.cumsum()


In [6]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
len(df)


71059

In [7]:
# now only keep the coloumn we needed from list of coloumns
df.columns


Index(['Index', 'batting_team', 'bowling_team', 'city', 'runs_left',
       'balls_left', 'wickets', 'target_runs', 'cur_run_rate', 'req_run_rate',
       'result', 'match_id'],
      dtype='str')

In [8]:
# since we need only  these coloums for our data so
# 'runs_left','balls_left', 'wickets', 'target_runs', 'cur_run_rate', 'req_run_rate',
#        'result'
data=df[['match_id','runs_left','balls_left', 'wickets', 'target_runs', 'cur_run_rate', 'req_run_rate','result']]
data


,match_id,runs_left,balls_left,wickets,target_runs,cur_run_rate,req_run_rate,result
0,1,207,119,10,208,6.000000,10.436975,0
1,1,207,118,10,208,3.000000,10.525424,0
2,1,207,117,10,208,2.000000,10.615385,0
3,1,205,116,10,208,4.500000,10.603448,0
4,1,201,115,10,208,8.400000,10.486957,0
...,...,...,...,...,...,...,...,...
72408,625,1,4,5,153,7.862069,1.500000,0
72409,625,0,3,5,153,7.897436,0.000000,0
72410,625,0,2,4,153,7.881356,0.000000,0
72411,625,0,1,4,153,7.915966,0.000000,0


In [9]:
# Use wickets_left as the primary feature (10 - wickets fallen)
data['wickets_left'] = 10 - data['wickets']

# Assuming balls_left counts remaining balls in innings
def match_phase(balls_left):
    if balls_left > 96:        # Overs 1-4 → balls 120-97
        return 'Powerplay'
    elif balls_left > 36:      # Overs 5-14 → balls 96-37
        return 'Middle'
    else:                      # Overs 15-20 → balls 36-0
        return 'Death'

data['phase'] = data['balls_left'].apply(match_phase)

data.head()


,match_id,runs_left,balls_left,wickets,target_runs,cur_run_rate,req_run_rate,result,wickets_left,phase
0,1,207,119,10,208,6.0,10.436975,0,0,Powerplay
1,1,207,118,10,208,3.0,10.525424,0,0,Powerplay
2,1,207,117,10,208,2.0,10.615385,0,0,Powerplay
3,1,205,116,10,208,4.5,10.603448,0,0,Powerplay
4,1,201,115,10,208,8.4,10.486957,0,0,Powerplay


In [10]:
phase_map = {'Powerplay':0, 'Middle':1, 'Death':2}
data['phase_num'] = data['phase'].map(phase_map)


In [11]:
# export csv as cleaned dataset
data.to_csv('cleaned_dataset.csv', index=False)



# Splitting data in test and train


In [12]:
data['pressure'] = data['req_run_rate'] - data['cur_run_rate']
data['balls_per_wicket'] = data['balls_left'] / (data['wickets'] + 1)
data['target_ratio'] = data['runs_left'] / (data['target_runs'] + 1)


In [13]:
# Additional feature calculation
print('Calculating advanced features...')

# 1. Momentum Features (Rolling windows per match)
data['runs_last_12'] = data.groupby('match_id')['runs_left'].diff(periods=-12).fillna(0)
data['wickets_last_18'] = data.groupby('match_id')['wickets'].diff(periods=18).fillna(0)

# 2. Resource Interaction (Non-linear value of wickets vs balls)
data['resource_score'] = (data['wickets_left']**1.5) * (data['balls_left']**0.5)

# 3. Run Rate Ratio (Relative difficulty)
data['rr_ratio'] = data['req_run_rate'] / (data['cur_run_rate'] + 0.5)

# 4. Venue Difficulty (Target vs City Average)
if 'city' not in data.columns:
    data['city'] = df.loc[data.index, 'city']

city_avg = data.groupby('city')['target_runs'].transform('mean')
data['venue_par_diff'] = data['target_runs'] - city_avg

# Update the feature list for the model
features = [
    'runs_left', 'balls_left', 'wickets_left', 
    'cur_run_rate', 'req_run_rate', 'pressure', 
    'resource_score', 'rr_ratio', 'runs_last_12', 
    'wickets_last_18', 'venue_par_diff'
]

print('New Feature List:', features)
data[features].head()


Calculating advanced features...
New Feature List: ['runs_left', 'balls_left', 'wickets_left', 'cur_run_rate', 'req_run_rate', 'pressure', 'resource_score', 'rr_ratio', 'runs_last_12', 'wickets_last_18', 'venue_par_diff']


,runs_left,balls_left,wickets_left,cur_run_rate,req_run_rate,pressure,resource_score,rr_ratio,runs_last_12,wickets_last_18,venue_par_diff
0,207,119,0,6.0,10.436975,4.436975,0.0,1.605688,11.0,0.0,42.531559
1,207,118,0,3.0,10.525424,7.525424,0.0,3.007264,15.0,0.0,42.531559
2,207,117,0,2.0,10.615385,8.615385,0.0,4.246154,16.0,0.0,42.531559
3,205,116,0,4.5,10.603448,6.103448,0.0,2.120690,18.0,0.0,42.531559
4,201,115,0,8.4,10.486957,2.086957,0.0,1.178310,15.0,0.0,42.531559


In [14]:

# Split by entire matches to prevent leakage
from sklearn.model_selection import train_test_split
unique_matches = data["match_id"].unique()
train_ids, test_ids = train_test_split(unique_matches, test_size=0.2, random_state=42)
train_data = data[data["match_id"].isin(train_ids)]
test_data = data[data["match_id"].isin(test_ids)]

X_train = train_data[features]
X_test  = test_data[features]
y_train = train_data["result"]
y_test = test_data["result"]

# Clean infinities / NaNs
import numpy as np
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_test.median())


In [15]:
len(X_train)


56787

In [16]:
len(X_test)


14272

In [17]:
# Checking for infinities
print(np.isinf(X_train).sum())

# Check for NaNs
print(X_train.isna().sum())


runs_left          0
balls_left         0
wickets_left       0
cur_run_rate       0
req_run_rate       0
pressure           0
resource_score     0
rr_ratio           0
runs_last_12       0
wickets_last_18    0
venue_par_diff     0
dtype: int64
runs_left          0
balls_left         0
wickets_left       0
cur_run_rate       0
req_run_rate       0
pressure           0
resource_score     0
rr_ratio           0
runs_last_12       0
wickets_last_18    0
venue_par_diff     0
dtype: int64


In [18]:
# # Replace infinite values in X_train and X_test before scaling
# from sklearn.preprocessing import StandardScaler
# # Replace infinite values only in 'req_run_rate' column
# X_train['req_run_rate'] = X_train['req_run_rate'].replace([np.inf, -np.inf], 40.0)
# X_test['req_run_rate'] = X_test['req_run_rate'].replace([np.inf, -np.inf], 40.0)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
# %pip install tensorflow


In [20]:
# --- TensorFlow/Keras Model (matching sklearn MLP) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

def build_model(input_shape):
    model = keras.Sequential([
        layers.Input(shape=(input_shape,)),
        
        # Layer 1: 128 neurons (matches sklearn)
        layers.Dense(128, activation='relu'),
        
        # Layer 2: 64 neurons
        layers.Dense(64, activation='relu'),
        
        # Layer 3: 64 neurons
        layers.Dense(64, activation='relu'),
        
        # Layer 4: 32 neurons
        layers.Dense(32, activation='relu'),
        
        # Output layer: sigmoid for binary classification
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),  # Same as sklearn 'adam'
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

model = build_model(X_train_scaled.shape[1])
model.summary()

# Train with similar settings to sklearn
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=500,  # Similar to max_iter=500
    batch_size=200,  # sklearn uses ~200 by default
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    ],
    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,065 (62.75 KB)

 Trainable params: 16,065 (62.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7909 - auc: 0.8794 - loss: 0.4293 - val_accuracy: 0.7446 - val_auc: 0.8501 - val_loss: 0.4817
Epoch 2/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8018 - auc: 0.8980 - loss: 0.3952 - val_accuracy: 0.7434 - val_auc: 0.8503 - val_loss: 0.4787
Epoch 3/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8095 - auc: 0.9037 - loss: 0.3832 - val_accuracy: 0.7372 - val_auc: 0.8457 - val_loss: 0.4997
Epoch 4/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8158 - auc: 0.9098 - loss: 0.3711 - val_accuracy: 0.7269 - val_auc: 0.8418 - val_loss: 0.5088
Epoch 5/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8230 - auc: 0.9154 - loss: 0.3606 - val_accuracy: 0.7355 - val_auc: 0.8385 - val_loss: 0.5327
Epoch 6/500
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8320 - auc: 0.9216 - loss: 0.3482 - val_accuracy: 0.7287 - val_auc: 0.8318 - val_loss: 0.5501
Epoch 7/500
228/228 ━━━━━━━━━━━━━━

In [21]:
# --- Keras Model Evaluation ---
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

# Predict probabilities
probs_keras = model.predict(X_test_scaled).flatten()

print('Keras Model Performance:')
print('Accuracy:', accuracy_score(y_test, probs_keras > 0.5))
print('AUC:', roc_auc_score(y_test, probs_keras))
print('LogLoss:', log_loss(y_test, probs_keras))


446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step
Keras Model Performance:
Accuracy: 0.8067544843049327
AUC: 0.8874345556143918
LogLoss: 0.4165357439022864


In [22]:
# # --- Save the Keras Model for Flutter ---
# # Save as TFLite for mobile deployment
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# tflite_model = converter.convert()

# with open('cricket_win_predictor.tflite', 'wb') as f:
#     f.write(tflite_model)

# print('Model saved as cricket_win_predictor.tflite')

# # Also save the scaler params for Flutter
# import json
# scaler_params = {
#     'mean': scaler.mean_.tolist(),
#     'std': scaler.scale_.tolist(),
#     'features': features
# }
# with open('scaler_params.json', 'w') as f:
#     json.dump(scaler_params, f, indent=2)

# print('Scaler params saved as scaler_params.json')


### Deep Learning Model (MLP) - More Layers and Epochs
Addressing the request for more complexity, layers, and epochs.

In [23]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

# Creating a model with multiple layers (3 layers: 128, 64, 32)
# Increasing max_iter (epochs) to 500
mlp = MLPClassifier(hidden_layer_sizes=(128,64, 64, 32), 
                    activation="relu", 
                    solver="adam", 
                    max_iter=10, 
                    random_state=42, 
                    verbose=True)

print("Training MLP Model...")
mlp.fit(X_train_scaled, y_train)

# Predict
probs_mlp = mlp.predict_proba(X_test_scaled)[:, 1]

print("\nMLP Performance:")
print("MLP Accuracy:", accuracy_score(y_test, probs_mlp > 0.5))
print("MLP AUC:", roc_auc_score(y_test, probs_mlp))
print("MLP LogLoss:", log_loss(y_test, probs_mlp))


Training MLP Model...
Iteration 1, loss = 0.43607260
Iteration 2, loss = 0.40615428
Iteration 3, loss = 0.39492186
Iteration 4, loss = 0.38404898
Iteration 5, loss = 0.37236118
Iteration 6, loss = 0.35959655
Iteration 7, loss = 0.34945847
Iteration 8, loss = 0.33784753
Iteration 9, loss = 0.32520516
Iteration 10, loss = 0.31556461

MLP Performance:
MLP Accuracy: 0.7778167040358744
MLP AUC: 0.8662755195175738
MLP LogLoss: 0.5271132489977894


C:\Users\Muhammad Abu Huraira\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


In [24]:
# %pip install xgboost scikit-learn

In [25]:
# from xgboost import XGBClassifier
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

# xgb = XGBClassifier(
#     n_estimators=1000,
#     learning_rate=0.01,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric='logloss',
# )

# # Calibrate probabilities
# xgb_cal = CalibratedClassifierCV(xgb, method='sigmoid', cv=5)
# xgb_cal.fit(X_train_scaled if 'X_train_scaled' in locals() else X_train, y_train)

# # Predict
# probs_xgb = xgb_cal.predict_proba(X_test_scaled if 'X_test_scaled' in locals() else X_test)[:,1]

# print("XGB LogLoss:", log_loss(y_test, probs_xgb))
# print("XGB Brier Score:", brier_score_loss(y_test, probs_xgb))
# print("XGB ROC AUC:", roc_auc_score(y_test, probs_xgb))


In [26]:
import pandas as pd
import numpy as np

# -------- raw match state ----------
runs_left = 24
balls_left = 23
wickets = 7
target_runs = 150
cur_run_rate = 10
venue_avg_target = 155  # Historical average at this venue

# -------- feature engineering (same logic as training) ----------
req_run_rate = runs_left / (balls_left/6 + 1e-9)
wickets_left = 10 - wickets
pressure = req_run_rate - cur_run_rate

# NEW FEATURES (must match training)
resource_score = (wickets_left ** 1.5) * (balls_left ** 0.5)
rr_ratio = req_run_rate / (cur_run_rate + 0.5)
runs_last_12 = 15  # Sum of runs in last 12 balls (example value)
wickets_last_18 = 1  # Wickets in last 18 balls (example value)
venue_par_diff = target_runs - venue_avg_target

# -------- Create sample DataFrame ----------
sample = pd.DataFrame([[
    runs_left, balls_left, wickets_left, cur_run_rate, req_run_rate,
    pressure, resource_score, rr_ratio, runs_last_12, wickets_last_18, venue_par_diff
]], columns=[
    'runs_left', 'balls_left', 'wickets_left', 'cur_run_rate', 'req_run_rate',
    'pressure', 'resource_score', 'rr_ratio', 'runs_last_12', 'wickets_last_18', 'venue_par_diff'
])

print(sample)

   runs_left  balls_left  wickets_left  cur_run_rate  req_run_rate  pressure  \
0         24          23             3            10       6.26087  -3.73913   

   resource_score  rr_ratio  runs_last_12  wickets_last_18  venue_par_diff  
0       24.919872  0.596273            15                1              -5  


In [27]:
# --- Single Sample Prediction using Both Models ---
sample_scaled = scaler.transform(sample)

# Use Keras model for prediction
prob_keras = model.predict(sample_scaled, verbose=0)[0][0]

print('=== Keras Model ===')
print('Win Probability:', round(prob_keras * 100, 2), '%')
print('Lose Probability:', round((1 - prob_keras) * 100, 2), '%')

# Use sklearn MLP for prediction
prob_mlp = mlp.predict_proba(sample_scaled)[0][1]

print('\n=== sklearn MLP ===')
print('Win Probability:', round(prob_mlp * 100, 2), '%')
print('Lose Probability:', round((1 - prob_mlp) * 100, 2), '%')

# Compare
print('\n=== Comparison ===')
print(f'Difference: {abs(prob_keras - prob_mlp) * 100:.2f}%')

=== Keras Model ===
Win Probability: 98.95 %
Lose Probability: 1.05 %

=== sklearn MLP ===
Win Probability: 99.92 %
Lose Probability: 0.08 %

=== Comparison ===
Difference: 0.97%


In [29]:
# prob_xgb = xgb_cal.predict_proba(sample)[0][1]

# print("XGB Win %:", round(prob_xgb*100,2))
# print("Lose %:", round((1-prob_xgb)*100,2))


# Checking data normal or not

In [30]:
# assume `data` is your full dataframe used to build X,y (before train/test split)
# and 'result' = 1 if chasing team won, 0 otherwise

# 1) How many extreme states exist?
extreme = data[(data['runs_left'] >= 10) & (data['balls_left'] <= 1)]
print("Extreme rows:", len(extreme))
print(extreme[['runs_left','balls_left','wickets','result']].value_counts()[:10])
print("Result counts in extreme states:\n", extreme['result'].value_counts(normalize=False))

# 2) Empirical win-rate table for small balls_left
grp = data.groupby(['balls_left','runs_left'])['result'].agg(['mean','count']).reset_index()
print(grp[(grp['balls_left']<=2) & (grp['runs_left']<=30)].sort_values(['balls_left','runs_left']).head(20))


Extreme rows: 50
runs_left  balls_left  wickets  result
11         1           1        0         5
12         1           5        0         4
13         1           3        0         4
11         1           2        0         2
14         1           4        0         2
10         1           1        0         2
15         1           1        0         2
10         1           2        0         2
15         1           3        0         2
13         1           4        0         2
Name: count, dtype: int64
Result counts in extreme states:
 result
0    50
Name: count, dtype: int64
    balls_left  runs_left      mean  count
0            0          0  0.777778     18
1            1          0  0.900000     30
2            1          1  0.833333      6
3            1          2  0.750000      8
4            1          3  0.777778      9
5            1          4  0.200000     10
6            1          5  0.500000      6
7            1          6  0.375000      8
8            1  

In [31]:
# Check relationship between runs_left=0 and result
zero_runs = data[data['runs_left']==0]
print("When runs_left == 0 (chase finished), result distribution:")
print(zero_runs['result'].value_counts(normalize=True))


When runs_left == 0 (chase finished), result distribution:
result
1    0.970745
0    0.029255
Name: proportion, dtype: float64


In [32]:
# --- Export sklearn MLP Weights for Manual Dart Implementation ---
import json
import numpy as np

# Assuming 'mlp' is your trained MLPClassifier
# Get the weights and biases
weights = [w.tolist() for w in mlp.coefs_]
biases = [b.tolist() for b in mlp.intercepts_]

mlp_params = {
    'weights': weights,
    'biases': biases,
    'activation': mlp.activation,
    'n_layers': mlp.n_layers_,
    'hidden_layer_sizes': list(mlp.hidden_layer_sizes),
    'classes': mlp.classes_.tolist()
}

# Also save scaler params
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
    'features': features
}

with open('mlp_sklearn_model.json', 'w') as f:
    json.dump({'model': mlp_params, 'scaler': scaler_params}, f, indent=2)

print('sklearn MLP exported to mlp_sklearn_model.json')
print(f'Architecture: {mlp.hidden_layer_sizes}')
print(f'Activation: {mlp.activation}')


sklearn MLP exported to mlp_sklearn_model.json
Architecture: (128, 64, 64, 32)
Activation: relu


In [33]:
# --- Export TensorFlow/Keras Model to TFLite ---
import tensorflow as tf
import json

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Optimize for mobile
tflite_model = converter.convert()

with open('keras_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Also save scaler params
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
    'features': features
}

with open('keras_scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)

print('TensorFlow model exported to keras_model.tflite')
print('Scaler params exported to keras_scaler_params.json')
print(f'TFLite model size: {len(tflite_model) / 1024:.2f} KB')


INFO:tensorflow:Assets written to: C:\Users\MUHAMM~1\AppData\Local\Temp\tmp_x8q5rdo\assets


INFO:tensorflow:Assets written to: C:\Users\MUHAMM~1\AppData\Local\Temp\tmp_x8q5rdo\assets


Saved artifact at 'C:\Users\MUHAMM~1\AppData\Local\Temp\tmp_x8q5rdo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1459121262992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121267408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121265488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121264912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121267984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121265680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121268368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121267216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121266832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1459121268176: TensorSpec(shape=(), dtype=tf.resource, name=None)
TensorFlow mo

In [34]:
#  now this cell will help to export model in onnx form so that it can be integrated in flutter application using fluuter onnxruntime package

In [40]:
%pip install skl2onnx

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: C:\Users\Muhammad Abu Huraira\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [41]:
# --- Direct Export sklearn MLP to ONNX ---
# %pip install skl2onnx

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx
import json

# Define input type (11 features)
initial_type = [('input', FloatTensorType([None, 11]))]

# Convert sklearn MLP directly to ONNX
onnx_model = convert_sklearn(mlp, initial_types=initial_type, target_opset=13)

# Save the ONNX model
onnx.save_model(onnx_model, 'cricket_model.onnx')

# Save scaler params
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
    'features': features
}

with open('onnx_scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)

print('✅ sklearn MLP exported to cricket_model.onnx')
print('✅ Scaler params exported to onnx_scaler_params.json')

# Verify
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('cricket_model.onnx')
input_name = sess.get_inputs()[0].name

test_input = X_test_scaled[:5].astype(np.float32)
onnx_output = sess.run(None, {input_name: test_input})

print(f'\n📊 Predictions (probabilities): {onnx_output[1][:5]}')

import os
print(f'📦 Model size: {os.path.getsize("cricket_model.onnx") / 1024:.2f} KB')

✅ sklearn MLP exported to cricket_model.onnx
✅ Scaler params exported to onnx_scaler_params.json

📊 Predictions (probabilities): [{0: 0.6068928241729736, 1: 0.393107146024704}, {0: 0.6092943549156189, 1: 0.3907056450843811}, {0: 0.5887056589126587, 1: 0.4112943410873413}, {0: 0.5715208649635315, 1: 0.4284791350364685}, {0: 0.5694422125816345, 1: 0.4305577874183655}]
📦 Model size: 64.72 KB
